In [1]:
# Install latest version from local files (only necessary after pulling updated version from Git).
# Restart Jupyter kernel when install has finished.
%pip install -e ..

Obtaining file:///home/maarten/Documents/Projects/BioPAP/BioPAP
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for BioPAP (pyproject.toml) ... done
  Created wheel for BioPAP: filename=biopap-0.0.1-0.editable-py3-none-any.whl size=3010 sha256=25c50387a9c76f787adfaf544a1c8c424698e87057ece2355d611c79bcd64692
  Stored in directory: /tmp/pip-ephem-wheel-cache-oyemupsm/wheels/a9/55/80/ba98fe05e3d05116737c29a1c667eb5bffec478c135f508c7b
Successfully built BioPAP
  Attempting uninstall: BioPAP
    Found existing installation: BioPAP 0.0.1
    Uninstalling BioPAP-0.0.1:
      Successfully uninstalled BioPAP-0.0.1
Note: you may need to restart the kernel to use updated packages.


In [2]:
# Some Windows installations require explicitly installing mne (required only once; restart Jupyter kernel after installing).
%pip install mne mne_bids

Note: you may need to restart the kernel to use updated packages.


In [3]:
from biopap.pipeline import PipelineStep, Pipeline

from biopap.modules.load_data import LoadData
from biopap.modules.anonymize import Anonymize
from biopap.modules.rereference import Rereference
from biopap.modules.set_channel_type_and_montage import SetChannelTypeAndMontage
from biopap.modules.eog_channels import EOGChannels
from biopap.modules.blink_detection import BlinkDetection
from biopap.modules.iir_filter import IIRFilter
from biopap.modules.downsample import Downsample
from biopap.modules.segmentation import Segmentation
from biopap.modules.apply_ica import ApplyICA
from biopap.modules.artifact_reject import ArtifactReject
from biopap.modules.baseline_correction import BaselineCorrection
from biopap.modules.save_output import SaveOutput
from biopap.modules.bids import BIDSConversion
from biopap.modules.plot import Plot

In [4]:
import mne
from pathlib import Path
from pprint import pprint

In [5]:
# To avoid issues with local settings and GitHub updates, create a small notebook
# called 'my_config.ipynb' to hold variables that are likely to change locally. 
# In the same folder as this file, there is an example notebook called
# 'example.my_config.ipynb' that can serve as a template.

config_notebook = './my_config.ipynb'

if Path(config_notebook).exists():
    %run {config_notebook}
else:
    data_file = '/biopap/data/example.bdf'
    config_file = './data/pipeline.yaml'

if not Path(config_file).exists():
    raise FileNotFoundError(f"Config file {config_file} does not exist.")

if not Path(data_file).exists():
    raise FileNotFoundError(f"Data file {data_file} does not exist.")

# BioPAP Pipeline Notebook

## Workflow Overview

This notebook supports a **two-phase workflow** for EEG preprocessing:

### Part 1: Exploratory
- Test individual modules with your data
- Tweak configurations to see their effects
- Inspect outputs after each processing step
- Choose which modules you need for your preprocessing

### Part 2: Structured Pipeline
- Define your pipeline using the `Pipeline` class
- Use the configuration file for reproducibility `./data/pipeline.yaml`
- Run the complete pipeline to check if it works before setting up batch processing


---

## Part 1: Exploratory

Test individual modules, inspect outputs, and determine the preprocessing steps for your project.

### About the modules

When instantiated, all processing modules receive the data to be processed as a RawEDF-type object, which is assigned to an attribute labeled `raw`. Once done processing, the processed data is available as an attribute labeled `processed` of the module. The processed data has the same format as the raw input, so the output of one module can become the input of the next, thus allowing you to easily chain modules together.

The only exception is the Load module, which reads the initial data from disk without any further processing, and therefore only has raw data.

This notebook demonstrates all available modules and their basic configurations. For detailed information about each module, its parameters, and recommended usage patterns, consult the [API documentation](../api/) and [tutorials](../tutorials/).

### Loading data

In [6]:
# Read the data file and load it into memory.
load = LoadData(file_path=data_file)

Extracting EDF parameters from /data/BioPAP/data/CBN1BD071212-Cropped.bdf...
BDF file detected
Setting channel info structure...
Creating raw.info structure...
Reading 0 ... 3276799  =      0.000 ...  1600.000 secs...


In [7]:
# Display basic info
load.raw.info

<Info | 8 non-empty values
 bads: []
 ch_names: Fp1, AF7, AF3, F1, F3, F5, F7, FT7, FC5, FC3, FC1, C1, C3, C5, ...
 chs: 79 EEG, 1 Stimulus
 custom_ref_applied: False
 highpass: 0.0 Hz
 lowpass: 417.0 Hz
 meas_date: 2012-12-07 12:14:08 UTC
 nchan: 80
 projs: []
 sfreq: 2048.0 Hz
 subject_info: <subject_info | his_id: CBN1BD071212>
>

In [8]:
# Show channel info
load.raw.describe()

<RawEDF | CBN1BD071212-Cropped.bdf, 80 x 3276800 (1600.0 s), ~1.95 GiB, data loaded>
ch  name    type  unit        min         Q1     median         Q3        max
 0  Fp1     EEG   µV     -2182.39   -1219.42    -918.55    -108.20     965.80
 1  AF7     EEG   µV       719.11    1136.45    1638.20    2268.07    3978.20
 2  AF3     EEG   µV       146.23    1396.14    2791.23    4102.60    5126.76
 3  F1      EEG   µV     12185.24   13145.21   13798.77   14291.90   14604.05
 4  F3      EEG   µV     10791.46   11679.20   12393.62   13127.18   13596.55
 5  F5      EEG   µV      3250.67    4102.23    4940.98    5700.66    6406.85
 6  F7      EEG   µV      9997.75   11560.90   13188.77   14673.77   15947.58
 7  FT7     EEG   µV      2811.79    4854.66    6670.91    8557.19    9902.78
 8  FC5     EEG   µV      7096.60    7824.53    8317.50    8796.56    9461.00
 9  FC3     EEG   µV      2982.57    4458.29    5286.44    5657.29    5930.57
10  FC1     EEG   µV     10697.43   11290.09   11819.06  

In [9]:
# Use the Plot-module to inspect the data
plt = Plot(raw=load.raw, title="Initial state")

Finding events on: Status
Trigger channel Status has a non-zero initial value of 65536 (consider using initial_event=True to detect this event)
Removing orphaned offset at the beginning of the file.
2042 events found on stim channel Status
Event IDs: [  1   2   3   4   5   6   9  10  11  12  13  14  15  16  17  18  19  20
  21  22  23  24  25  26  27  28  29  30  31  32  33  34  35  36  37  38
  39  40  41  42  43  44  45  46  47  48  49  50  51  52  53  54  55  56
  57  58  59  60  61  62  63  64  65  66  67  68  69  96  97  98  99 101
 102 103 104 105 106 121 122 123 124 125 126 141 142 143 144 145 146 161
 162 163 164 165 201 203 211 212 213 231 232 233 240 241 253]


2026-08-28 14:17:41,375:INFO:acceleratesupport:No OpenGL_accelerate module loaded: No module named 'OpenGL_accelerate'


Using qt as 2D backend.


/home/maarten/anaconda3/envs/biopap/lib/python3.11/site-packages/mne_qt_browser/_pg_figure.py:1544: RuntimeWarning: Precision loss occurred in moment calculation due to catastrophic cancellation. This occurs when the data are nearly identical. Results may be unreliable.
  z = zscore(data, axis=1)


Channels marked as bad:
none


### Anonymization

In [10]:
# Anonymize the data: removes personal identifiers and shifts recording date
# - 'daysback' defines the number of days subtracted from the original measurement date in the data. Its value can either be a 
#   positive integer, or the word 'random', in which case the program will subtract a random number between 1 and 100.
# - 'experimenter' and 'participant_id' are both optional.

ano = Anonymize(raw=load.raw, daysback=100, experimenter='J. Jansen', participant_id='AB00123')
ano.processed.info

<Info | 10 non-empty values
 bads: []
 ch_names: Fp1, AF7, AF3, F1, F3, F5, F7, FT7, FC5, FC3, FC1, C1, C3, C5, ...
 chs: 79 EEG, 1 Stimulus
 custom_ref_applied: False
 description: Anonymized using a time shift to preserve age at acquisition
 experimenter: J. Jansen
 highpass: 0.0 Hz
 lowpass: 417.0 Hz
 meas_date: 2012-08-29 12:14:08 UTC
 nchan: 80
 projs: []
 sfreq: 2048.0 Hz
 subject_info: <subject_info | his_id: AB00123>
>

### Rereferencing

In [11]:
ref = Rereference(raw=ano.processed, ref_channels=['EXG1', 'EXG2'])
ref.processed.info

EEG channel type selected for re-referencing
Applying a custom ('EEG',) reference.


<Info | 11 non-empty values
 bads: []
 ch_names: Fp1, AF7, AF3, F1, F3, F5, F7, FT7, FC5, FC3, FC1, C1, C3, C5, ...
 chs: 79 EEG, 1 Stimulus
 custom_ref_applied: True
 description: Anonymized using a time shift to preserve age at acquisition
 experimenter: J. Jansen
 highpass: 0.0 Hz
 lowpass: 417.0 Hz
 meas_date: 2012-08-29 12:14:08 UTC
 nchan: 80
 projs: []
 sfreq: 2048.0 Hz
 subject_info: <subject_info | his_id: AB00123>
>

### Create EOG channels

In [12]:
# Create horizontal and vertical EOG channels from electrode pairs
eog = EOGChannels(
    raw=ref.processed,
    ref_channels=['hEOG', 'vEOG'],
    anode_channels=['EXG3','EXG5'],
    cathode_channels=['EXG4','EXG6'])

eog.processed.info

EEG channel type selected for re-referencing
Creating RawArray with float64 data, n_channels=2, n_times=3276800
    Range : 0 ... 3276799 =      0.000 ...  1600.000 secs
Ready.
Added the following bipolar channels:
hEOG, vEOG


<Info | 12 non-empty values
 bads: []
 ch_names: Fp1, AF7, AF3, F1, F3, F5, F7, FT7, FC5, FC3, FC1, C1, C3, C5, ...
 chs: 79 EEG, 1 Stimulus, 2 EOG
 custom_ref_applied: True
 description: Anonymized using a time shift to preserve age at acquisition
 dig: 0 items
 experimenter: J. Jansen
 highpass: 0.0 Hz
 lowpass: 417.0 Hz
 meas_date: 2012-08-29 12:14:08 UTC
 nchan: 82
 projs: []
 sfreq: 2048.0 Hz
 subject_info: <subject_info | his_id: AB00123>
>

### Set Channel Types and Montage

In [13]:
# Define channel types mapping the original channel names to MNE types
# Adjust the channel names and montage based on your actual data
channel_types = {
        'misc': ['EXG1','EXG2','EXG3','EXG4','EXG5','EXG6','EXG7','EXG8','Erg1','Erg2','Plet'],
        'gsr': ['GSR1', 'GSR2'],
        'resp': ['Resp'],
        'temperature': ['Temp']
}

montage = SetChannelTypeAndMontage(
    raw=eog.processed,
    channel_types=channel_types,
    mne_standard_montage='biosemi64'
)
montage.processed.info

/home/maarten/Documents/Projects/BioPAP/BioPAP/src/biopap/modules/set_channel_type_and_montage.py:70: RuntimeWarning: The unit for channel(s) EXG1, EXG2, EXG3, EXG4, EXG5, EXG6, EXG7, EXG8, Erg1, Erg2, Plet has changed from V to NA.
  self.processed.set_channel_types({chs: type for chs in channel_types.get(type)}, **s_kwargs)
/home/maarten/Documents/Projects/BioPAP/BioPAP/src/biopap/modules/set_channel_type_and_montage.py:70: RuntimeWarning: The unit for channel(s) GSR1, GSR2 has changed from V to S.
  self.processed.set_channel_types({chs: type for chs in channel_types.get(type)}, **s_kwargs)
/home/maarten/Documents/Projects/BioPAP/BioPAP/src/biopap/modules/set_channel_type_and_montage.py:70: RuntimeWarning: The unit for channel(s) Temp has changed from V to C.
  self.processed.set_channel_types({chs: type for chs in channel_types.get(type)}, **s_kwargs)


<Info | 12 non-empty values
 bads: []
 ch_names: Fp1, AF7, AF3, F1, F3, F5, F7, FT7, FC5, FC3, FC1, C1, C3, C5, ...
 chs: 64 EEG, 11 misc, 2 Galvanic skin response, 1 Respiration monitoring channel, 1 Temperature, 1 Stimulus, 2 EOG
 custom_ref_applied: True
 description: Anonymized using a time shift to preserve age at acquisition
 dig: 67 items (3 Cardinal, 64 EEG)
 experimenter: J. Jansen
 highpass: 0.0 Hz
 lowpass: 417.0 Hz
 meas_date: 2012-08-29 12:14:08 UTC
 nchan: 82
 projs: []
 sfreq: 2048.0 Hz
 subject_info: <subject_info | his_id: AB00123>
>

In [14]:
# See available montages with:
mne.channels.get_builtin_montages()

['standard_1005',
 'standard_1020',
 'standard_alphabetic',
 'standard_postfixed',
 'standard_prefixed',
 'standard_primed',
 'biosemi16',
 'biosemi32',
 'biosemi64',
 'biosemi128',
 'biosemi160',
 'biosemi256',
 'easycap-M1',
 'easycap-M10',
 'easycap-M43',
 'EGI_256',
 'GSN-HydroCel-32',
 'GSN-HydroCel-64_1.0',
 'GSN-HydroCel-65_1.0',
 'GSN-HydroCel-128',
 'GSN-HydroCel-129',
 'GSN-HydroCel-256',
 'GSN-HydroCel-257',
 'mgh60',
 'mgh70',
 'artinis-octamon',
 'artinis-brite23',
 'brainproducts-RNP-BA-128']

### Filtering (IIR Filter)

In [15]:
# Apply IIR band-pass filter to remove low and high frequency noise
flt = IIRFilter(
    raw=montage.processed,
    l_freq=0.1,
    h_freq=35)

# IIRFilter has two additional paramaters:"
# - order (defaults to 4)
# - ftype (which defautts to 'butter')

Filtering raw data in 1 contiguous segment
Setting up band-pass filter from 0.1 - 35 Hz

IIR filter parameters
---------------------
Butterworth bandpass zero-phase (two-pass forward and reverse) non-causal filter:
- Filter order 16 (effective, after forward-backward)
- Cutoffs at 0.10, 35.00 Hz: -6.02, -6.02 dB



In [16]:
# Optionally, save a plot of the current state by specifying save folder. The screenshot will take the basename of the original
# source file, optionally taking a postfix to distinguish between multiple screenshots on separate points in the pipeline.
# This step is not interactive, so can also be used in a pipeline intended to run unattended.
# N.B., showing the interactive plot and saving cannot be done at the same time, but you can add multiple subsequent Plot-steps.
_ = Plot(raw=flt.processed, save_folder='./', save_postfix='IRR')

Finding events on: Status
Trigger channel Status has a non-zero initial value of 65536 (consider using initial_event=True to detect this event)
Removing orphaned offset at the beginning of the file.
2042 events found on stim channel Status
Event IDs: [  1   2   3   4   5   6   9  10  11  12  13  14  15  16  17  18  19  20
  21  22  23  24  25  26  27  28  29  30  31  32  33  34  35  36  37  38
  39  40  41  42  43  44  45  46  47  48  49  50  51  52  53  54  55  56
  57  58  59  60  61  62  63  64  65  66  67  68  69  96  97  98  99 101
 102 103 104 105 106 121 122 123 124 125 126 141 142 143 144 145 146 161
 162 163 164 165 201 203 211 212 213 231 232 233 240 241 253]
wrote plot to 'AB00123_plot_IRR.png'


### Downsampling

In [17]:
# Downsample the data to reduce file size and computational requirements
dwn = Downsample(
    raw=flt.processed,
    sfreq=500)  # Downsample to 500 Hz

# Additional parameter:
# - resample_events (default: True)

dwn.processed.info

Finding events on: Status
Trigger channel Status has a non-zero initial value of 65536 (consider using initial_event=True to detect this event)
Removing orphaned offset at the beginning of the file.
2042 events found on stim channel Status
Event IDs: [  1   2   3   4   5   6   9  10  11  12  13  14  15  16  17  18  19  20
  21  22  23  24  25  26  27  28  29  30  31  32  33  34  35  36  37  38
  39  40  41  42  43  44  45  46  47  48  49  50  51  52  53  54  55  56
  57  58  59  60  61  62  63  64  65  66  67  68  69  96  97  98  99 101
 102 103 104 105 106 121 122 123 124 125 126 141 142 143 144 145 146 161
 162 163 164 165 201 203 211 212 213 231 232 233 240 241 253]
Finding events on: Status
Trigger channel Status has a non-zero initial value of 65536 (consider using initial_event=True to detect this event)
Removing orphaned offset at the beginning of the file.
2041 events found on stim channel Status
Event IDs: [  1   2   3   4   5   6   9  10  11  12  13  14  15  16  17  18  19  2

/home/maarten/Documents/Projects/BioPAP/BioPAP/src/biopap/modules/downsample.py:52: RuntimeWarning: Resampling of the stim channels caused event information to become unreliable. Consider finding events on the original data and passing the event matrix as a parameter.
  self.processed.resample(** {


<Info | 12 non-empty values
 bads: []
 ch_names: Fp1, AF7, AF3, F1, F3, F5, F7, FT7, FC5, FC3, FC1, C1, C3, C5, ...
 chs: 64 EEG, 11 misc, 2 Galvanic skin response, 1 Respiration monitoring channel, 1 Temperature, 1 Stimulus, 2 EOG
 custom_ref_applied: True
 description: Anonymized using a time shift to preserve age at acquisition
 dig: 67 items (3 Cardinal, 64 EEG)
 experimenter: J. Jansen
 highpass: 0.1 Hz
 lowpass: 35.0 Hz
 meas_date: 2012-08-29 12:14:08 UTC
 nchan: 82
 projs: []
 sfreq: 500.0 Hz
 subject_info: <subject_info | his_id: AB00123>
>

### Blink detection

In [18]:
# Automatically detect blinks from EOG channels
blink = BlinkDetection(raw=dwn.processed)

# Use the Plot-module see the blink annotations
_ = Plot(raw=blink.processed, title="Show blink annotations")

Using EOG channels: hEOG, vEOG
EOG channel index for this subject is: [80 81]
Filtering the data to remove DC offset to help distinguish blinks from saccades
Selecting channel vEOG for blink detection
Setting up band-pass filter from 1 - 10 Hz

FIR filter parameters
---------------------
Designing a two-pass forward and reverse, zero-phase, non-causal bandpass filter:
- Windowed frequency-domain design (firwin2) method
- Hann window
- Lower passband edge: 1.00
- Lower transition bandwidth: 0.50 Hz (-12 dB cutoff frequency: 0.75 Hz)
- Upper passband edge: 10.00 Hz
- Upper transition bandwidth: 0.50 Hz (-12 dB cutoff frequency: 10.25 Hz)
- Filter length: 5000 samples (10.000 s)

Now detecting blinks and generating corresponding events
Found 235 significant peaks
Number of EOG events detected: 235
Finding events on: Status
Trigger channel Status has a non-zero initial value of 65536 (consider using initial_event=True to detect this event)
Removing orphaned offset at the beginning of the f

### Segmentation (Epoching)

In [19]:
# Create epochs around events/markers detected on the stimulation channel
# Adjust these parameters based on your task and stimulus markers
seg = Segmentation(
    raw=blink.processed,  
    stim_channel='Status',  
    start=-0.2,  
    end=0.8,     
    baseline=(None, 0),
    reference_markers={'range': '1-69'}
)

seg.processed

Finding events on: Status
Trigger channel Status has a non-zero initial value of 65536 (consider using initial_event=True to detect this event)
Removing orphaned offset at the beginning of the file.
2041 events found on stim channel Status
Event IDs: [  1   2   3   4   5   6   9  10  11  12  13  14  15  16  17  18  19  20
  21  22  23  24  25  26  27  28  29  30  31  32  33  34  35  36  37  38
  39  40  41  42  43  44  45  46  47  48  49  50  51  52  53  54  55  56
  57  58  59  60  61  62  63  64  65  66  67  68  69  96  97  98  99 101
 102 103 104 105 106 121 122 123 124 125 126 141 142 143 144 145 146 161
 162 163 164 165 201 203 211 212 213 231 232 233 240 241 253]
Not setting metadata
540 matching events found
Setting baseline interval to [-0.2, 0.0] s
Applying baseline correction (mode: mean)
0 projection items activated
Using data from preloaded Raw for 540 events and 501 original time points ...
20 bad epochs dropped


<Epochs | 520 events (all good), -0.2 – 0.8 s (baseline -0.2 – 0 s), ~163.1 MiB, data loaded,
 '1': 28
 '2': 29
 '3': 28
 '4': 29
 '5': 30
 '6': 30
 '9': 177
 '10': 3
 '11': 3
 '12': 2
 and 57 more events ...>

### Artifact Rejection

In [20]:
# Reject epochs with defined limits
# This identifies bad channels based on voltage criteria and either interpolates or replaces them with NaNs
artifact = ArtifactReject(
    raw=seg.processed,
    mode='interpolation',
    max_gradient_uv=50,
    max_amplitude_uv=75,
    min_activity_uv=0.5,
    interval_ms=100
)

Fp1: bad in 3 epochs
AF7: bad in 2 epochs
AF3: bad in 1 epochs
F1: bad in 0 epochs
F3: bad in 0 epochs
F5: bad in 0 epochs
F7: bad in 0 epochs
FT7: bad in 0 epochs
FC5: bad in 0 epochs
FC3: bad in 0 epochs
FC1: bad in 0 epochs
C1: bad in 0 epochs
C3: bad in 0 epochs
C5: bad in 0 epochs
T7: bad in 0 epochs
TP7: bad in 0 epochs
CP5: bad in 0 epochs
CP3: bad in 0 epochs
CP1: bad in 0 epochs
P1: bad in 0 epochs
P3: bad in 0 epochs
P5: bad in 0 epochs
P7: bad in 0 epochs
P9: bad in 0 epochs
PO7: bad in 0 epochs
PO3: bad in 0 epochs
O1: bad in 1 epochs
Iz: bad in 0 epochs
Oz: bad in 0 epochs
POz: bad in 0 epochs
Pz: bad in 3 epochs
CPz: bad in 0 epochs
Fpz: bad in 3 epochs
Fp2: bad in 3 epochs
AF8: bad in 1 epochs
AF4: bad in 0 epochs
AFz: bad in 0 epochs
Fz: bad in 0 epochs
F2: bad in 0 epochs
F4: bad in 0 epochs
F6: bad in 0 epochs
F8: bad in 0 epochs
FT8: bad in 0 epochs
FC6: bad in 0 epochs
FC4: bad in 0 epochs
FC2: bad in 0 epochs
FCz: bad in 0 epochs
Cz: bad in 0 epochs
C2: bad in 0 ep

### Baseline Correction

In [21]:
# Apply baseline correction to epochs
# The baseline period is specified as (start, end) in seconds relative to event
baseline = BaselineCorrection(
    raw=artifact.processed,
    baseline=(-0.2, 0)
)


Applying baseline correction (mode: mean)


### Independent Component Analysis (ICA)

In [22]:
# Apply ICA to decompose data and remove additional artifact components
# Use this as an alternative or in addition to Artifact Rejection and Blink Detection
ica_result = ApplyICA(
    raw=baseline.processed,
    n_components=None, # use all components
    method='fastica',
    silent=False
)

Fitting ICA to data using 64 channels (please be patient, this may take a while)


/home/maarten/Documents/Projects/BioPAP/BioPAP/src/biopap/modules/apply_ica.py:51: RuntimeWarning: The epochs you passed to ICA.fit() were baseline-corrected. However, we suggest to fit ICA only on data that has been high-pass filtered, but NOT baseline-corrected.
  ica = mne.preprocessing.ICA(n_components=n_components, method=method, **i_kwargs).fit(self.processed, **f_kwargs)


Selecting by non-zero PCA components: 64 components
Fitting ICA took 25.4s.
Applying ICA to Epochs instance
    Transforming to ICA space (64 components)
    Zeroing out 0 ICA components
    Projecting back using 64 PCA components


/home/maarten/Documents/Projects/BioPAP/BioPAP/src/biopap/modules/apply_ica.py:52: RuntimeWarning: The data you passed to ICA.apply() was baseline-corrected. Please note that ICA can introduce DC shifts, therefore you may wish to consider baseline-correcting the cleaned data again.
  self.processed = ica.apply(self.processed, **a_kwargs)


### Visualization with Plot

In [23]:
# Visualize the data before further processing or saving
plt = Plot(raw=ica_result.processed, title="Preprocessed data")

You seem to have overlapping epochs. Some event lines may be duplicated in the plot.


/home/maarten/Documents/Projects/BioPAP/BioPAP/src/biopap/modules/plot.py:175: RuntimeWarning: More events than default colors available. You should pass a list of unique colors.
  _ = self.processed.plot(** args | { 'block': True, 'show': True } | kwargs)


Dropped 0 epochs: 
The following epochs were marked as bad and are dropped:
[]
Channels marked as bad:
none


### Saving Output

In [24]:
# Save the processed data in various formats
# FIF format (MNE's native format)
save_output = SaveOutput(
    raw=ica_result.processed,
    export_formats='fif',
    output_file='./processed_data-epo.fif',
    overwrite=True
)

wrote 'processed_data-epo__PARTIAL.fif' (85,479,520)


/home/maarten/Documents/Projects/BioPAP/BioPAP/src/biopap/modules/save_output.py:108: RuntimeWarning: This filename (processed_data-epo__PARTIAL.fif) does not conform to MNE naming conventions. All epochs files should end with -epo.fif, -epo.fif.gz, _epo.fif or _epo.fif.gz
  self.processed.save(f_new, overwrite=overwrite)


### BIDS Conversion

In [25]:
# (OPTIONAL) Convert raw data to BIDS (Brain Imaging Data Structure) format
# BIDS conversion should be done on the raw, unprocessed data to preserve original parameters
# You'll need to prepare an events file with your marker definitions
# The events file should be a CSV with columns: ID and label

# Example of BIDS conversion (comment out if not needed)
# bids_convert = BIDSConversion(
#     raw=load.raw,  # Use the original raw data, NOT processed data
#     eventsfile='./data/events_metadata.csv',  # Path to your events/markers file
#     bids_root='./bids',  # Root directory for BIDS dataset
#     subject='001',  # Subject identifier
#     session='01',  # Session identifier
#     task='default_task',  # Task name
#     run=1,  # Run number
#     overwrite=False,
#     dataset_description={
#         'Name': 'BioPAP Example Dataset',
#         'BIDSVersion': '1.10.0',
#         'DatasetType': 'raw'
#     }
# )
# print("Data converted and saved to BIDS format")
# print(f"BIDS root: ./bids")


---

## Part 2: Structured Pipeline with Configuration Files

Part 2 uses the `Pipeline` class with a configuration file for reproducibility and batch processing.

The `Pipeline` class provides a structured way to:
- Load configurations from a file
- Add processing steps in order
- Run all steps sequentially without manual intervention
- Easily reuse the same pipeline on multiple datasets

### About Configuration Files

Configuration files (typically `pipeline.yaml`) store all module parameters in one place. This is useful for **batch processing**, to have the same set of parameters to process dataset identically. 

Example configuration file location: `./data/pipeline.yaml`

### Creating a Pipeline Instance

Create a new pipeline by instantiating the `Pipeline` class with your data and configuration file:


In [26]:
# Create a new pipeline, loading the raw data and the configuration file
pipe = Pipeline(data=load.raw, config=config_file, verbose='ERROR')

In [27]:
# Inspect the loaded configuration
print("Pipeline Configuration:")
pprint(pipe.config)

Pipeline Configuration:
{'Anonymize': {'daysback': 'random',
               'experimenter': 'J. Jansen',
               'participant_id': 'AB00123'},
 'ApplyICA': {'method': 'fastica'},
 'ArtifactReject': {'interval_ms': 100,
                    'max_amplitude_uv': 75,
                    'max_gradient_uv': 50,
                    'min_activity_uv': 0.5,
                    'mode': 'interpolation'},
 'BIDSConversion': {'bids_root': '/data/BioPAP/config/bids',
                    'dataset_description': {'acknowledgements': '',
                                            'authors': 'Marijn Struiksma',
                                            'data_license': 'CC-BY-4.0',
                                            'dataset_type': 'raw',
                                            'doi': 'doi:',
                                            'funding': 'NWO 277-89-001',
                                            'how_to_acknowledge': '',
                                            'name':

### Adding Steps to the Pipeline

A pipeline is made up of processing steps executed in order. Each step's output becomes the next step's input.

**Key points:**
- Do NOT specify input data when adding steps—the pipeline automatically chains outputs
- Settings are read from the configuration file by default
- Steps are not executed until you call `pipe.run()`
- You can reset the pipeline and rebuild it if needed

#### Example 1: Add basic steps with default configuration

In [28]:
# Add processing steps in order
pipe.add_step(Anonymize)
pipe.add_step(Downsample)
pipe.add_step(IIRFilter)
pipe.add_step(Plot)

# Print pipeline overview (steps and their configuration)
print("Pipeline Overview:")
print(*pipe.steps, sep='\n')

Pipeline Overview:
0: Anonymize ← {'daysback': 'random', 'experimenter': 'J. Jansen', 'participant_id': 'AB00123'}
1: Downsample ← {'sfreq': 500}
2: IIRFilter ← {'l_freq': 0.1, 'h_freq': 35}
3: Plot ← {}


#### Example 2: Using configuration labels for same module with different settings

When you need to run the same module multiple times with different configurations, use a `cfg_label` to distinguish them in the config file:


In [29]:
# Example: Use Downsample twice with different config labels
pipe.add_step(Downsample, cfg_label='Downsample_500Hz')
pipe.add_step(Downsample, cfg_label='Downsample_1000Hz')  # Different settings in config file

print("Pipeline with multiple Downsample steps:")
print(*pipe.steps, sep='\n')

Pipeline with multiple Downsample steps:
0: Anonymize ← {'daysback': 'random', 'experimenter': 'J. Jansen', 'participant_id': 'AB00123'}
1: Downsample ← {'sfreq': 500}
2: IIRFilter ← {'l_freq': 0.1, 'h_freq': 35}
3: Plot ← {}
4: Downsample ← {}
5: Downsample ← {}


#### Example 3: Override configuration parameters on-the-fly

You can override configuration file settings by passing a `cfg` dictionary:


In [30]:
# Example: Override configuration with custom parameters
pipe.add_step(IIRFilter, cfg={'h_freq': 50})  # Use 50 Hz instead of config file value

print("IIRFilter with overridden frequency:")
print(*pipe.steps, sep='\n')

IIRFilter with overridden frequency:
0: Anonymize ← {'daysback': 'random', 'experimenter': 'J. Jansen', 'participant_id': 'AB00123'}
1: Downsample ← {'sfreq': 500}
2: IIRFilter ← {'l_freq': 0.1, 'h_freq': 35}
3: Plot ← {}
4: Downsample ← {}
5: Downsample ← {}
6: IIRFilter ← {'l_freq': 0.1, 'h_freq': 50}


#### Resetting the Pipeline

To clear the pipeline and start over with a different configuration, use `reset()`:


In [31]:
# Clear the pipeline to start fresh
pipe.reset()
print("Pipeline has been reset")

Pipeline has been reset


### Complete Pipeline Example

Build a complete preprocessing pipeline with the recommended processing order:


In [32]:
# Reset pipeline for a fresh configuration
pipe.reset()

# Add all processing steps in the recommended order
pipe.add_step(Anonymize)
pipe.add_step(Rereference)
pipe.add_step(SetChannelTypeAndMontage)
pipe.add_step(EOGChannels)
pipe.add_step(IIRFilter)
pipe.add_step(Downsample)
pipe.add_step(BlinkDetection)
pipe.add_step(Segmentation)
pipe.add_step(ArtifactReject)
pipe.add_step(BaselineCorrection)
# pipe.add_step(ApplyICA)
pipe.add_step(SaveOutput)

### Inspect the  pipeline

In [33]:
# Display the complete pipeline
pipe.inspect()

00. Anonymize
    daysback: 'random'
    experimenter: 'J. Jansen'
    participant_id: 'AB00123'

01. Rereference *
    ref_channels: ['EXG1', 'EXG2']

02. SetChannelTypeAndMontage *
    channel_types:
      misc: ['EXG1', 'EXG2', 'EXG3', 'EXG4', 'EXG5', 'EXG6', 'EXG7', 'EXG8', 'Erg1', 'Erg2', 'Plet']
      gsr: ['GSR1', 'GSR2']
      resp: ['Resp']
      temperature: ['Temp']
    mne_standard_montage: 'biosemi64'

03. EOGChannels *
    ref_channels: ['hEOG', 'vEOG']
    anode_channels: ['EXG3', 'EXG5']
    cathode_channels: ['EXG4', 'EXG6']

04. IIRFilter *
    l_freq: 0.1
    h_freq: 35

05. Downsample *
    sfreq: 500

06. BlinkDetection *

07. Segmentation *
    reference_markers:
      list: None
      range: '10-69'
    stim_channel: 'Status'
    start: -0.2
    end: 1

08. ArtifactReject *
    mode: 'interpolation'
    max_gradient_uv: 50
    max_amplitude_uv: 75
    min_activity_uv: 0.5
    interval_ms: 100

09. BaselineCorrection *
    baseline: [-0.2, 0]

10. SaveOutput
    e

### Running the Pipeline

Once you've added all the desired steps, execute the entire pipeline with a single command:


In [34]:
# Run the entire pipeline sequentially
pipe.run()

print("Pipeline execution complete!")

2026-08-28 14:18:42,770:INFO:pipeline:Running (0) Anonymize [CBN1BD071212-Cropped]
2026-08-28 14:18:42,772:INFO:pipeline:Running (1) Rereference < {'ref_channels': ['EXG1', 'EXG2']} [CBN1BD071212-Cropped]
2026-08-28 14:18:42,925:INFO:pipeline:Running (2) SetChannelTypeAndMontage < {'channel_types': {'misc': ['EXG1', 'EXG2', 'EXG3', 'EXG4', 'EXG5', 'EXG6', 'EXG7', 'EXG8', 'Erg1', 'Erg2', 'Plet'], 'gsr': ['GSR1', 'GSR2'], 'resp': ['Resp'], 'temperature': ['Temp']}, 'mne_standard_montage': 'biosemi64'} [CBN1BD071212-Cropped]
2026-08-28 14:18:42,931:INFO:pipeline:Running (3) EOGChannels < {'ref_channels': ['hEOG', 'vEOG'], 'anode_channels': ['EXG3', 'EXG5'], 'cathode_channels': ['EXG4', 'EXG6']} [CBN1BD071212-Cropped]
2026-08-28 14:18:43,158:INFO:pipeline:Running (4) IIRFilter < {'l_freq': 0.1, 'h_freq': 35} [CBN1BD071212-Cropped]
2026-08-28 14:18:44,332:INFO:pipeline:Running (5) Downsample < {'sfreq': 500} [CBN1BD071212-Cropped]
2026-08-28 14:18:44,333:INFO:pipeline:Running (6) BlinkDetec

Pipeline execution complete!
